# Gerenciar pacotes

## instalar pacotes via pip se necessário

In [ ]:
%pip install geopandas -q

## importar pacotes

In [6]:
import pandas as pd
import geopandas as gpd
from glob import glob
import os

# Funções de trabalho

In [5]:
def area_calc(gdf, crs=None, limiar=10):
  '''
  Calcula a área em hectares de cada feição em um GeoDataFrame e adiciona colunas
  para limites superior e inferior da área baseados em um limiar percentual.

  Parâmetros:
  ----------
  gdf : geopandas.GeoDataFrame
      O GeoDataFrame de entrada contendo as geometrias para cálculo da área.
  crs : str, opcional
      O sistema de referência de coordenadas (CRS) para projetar o GeoDataFrame
      antes do cálculo da área. Se `None`, usa 'Albers IBGE SIRGAS 2000' por padrão.
      Deve ser um CRS projetado para cálculos de área precisos.
  limiar : int ou float, opcional
      O limiar percentual (0-100) usado para calcular as áreas máximas e mínimas.
      Por padrão é 10 (10%).

  Retorna:
  -------
  geopandas.GeoDataFrame
      Um GeoDataFrame com as colunas 'area_ha', 'area_max' e 'area_min' adicionadas.
  '''
  # Verifica se um CRS específico foi fornecido
  if crs is None:
    # Define o WKT para a projeção 'Albers IBGE SIRGAS 2000' se nenhum CRS for especificado
    wkt = 'PROJCS["Albers_IBGE_SIRGAS2000",\
    GEOGCS["GCS_SIRGAS_2000",\
        DATUM["D_SIRGAS_2000",\
            SPHEROID["GRS_1980",6378137.0,298.257222101]],\
        PRIMEM["Greenwich",0.0],\
        UNIT["Degree",0.0174532925199433]],\
    PROJECTION["Albers_Conic_Equal_Area"],\
    PARAMETER["False_Easting",5000000.0],\
    PARAMETER["False_Northing",10000000.0],\
    PARAMETER["central_meridian",-54.0],\
    PARAMETER["standard_parallel_1",-12.5],\
    PARAMETER["standard_parallel_2",-22.5],\
    PARAMETER["latitude_of_origin",-32.0],\
    UNIT["Meter",1.0]]'

    # Projeta o GeoDataFrame para o CRS definido
    gdf_crs = gdf.to_crs(wkt)
    print("Projeção utilizada: Albers IBGE SIRGAS 2000")

  else:
    # Projeta o GeoDataFrame para o CRS fornecido
    gdf_crs = gdf.to_crs(crs)
    # Tenta verificar se o CRS é projetado; se falhar, assume que não é planar
    try:
      gdf_crs.crs.is_projected
    except:
      print(f"A projeção {gdf_crs.crs} não é planar (é geográfica/angular).")
    else:
      print(f"Projeção utilizada: {gdf_crs.crs}")

  # Calcula a área de cada feição em metros quadrados e converte para hectares
  gdf_crs["area_ha"] = gdf_crs.area / 10000
  # Converte o limiar percentual para um fator decimal
  perc =  (limiar/100)
  # Calcula a área máxima permitida com base no limiar
  gdf_crs["area_max"] = gdf_crs['area_ha'] * (1 + perc)
  # Calcula a área mínima permitida com base no limiar
  gdf_crs["area_min"] = gdf_crs['area_ha'] * (1 - perc)

  print(f"Área calculada em hectares e limiares máximos e mínimos aplicados com {limiar}% de variação.")

  return gdf_crs

In [25]:
def unify_geometries(gdf):
  '''
  Unifica todas as geometrias de um GeoDataFrame em uma única geometria.
  A função mantém os atributos da primeira feição do GeoDataFrame original
  e substitui sua geometria pela união de todas as geometrias.

  Parâmetros:
  ----------
  gdf : geopandas.GeoDataFrame
      O GeoDataFrame de entrada com as geometrias a serem unificadas.

  Retorna:
  -------
  geopandas.GeoDataFrame
      Um novo GeoDataFrame contendo uma única linha com a geometria unificada
      e os atributos da primeira linha do GeoDataFrame original.
  '''
  # Cria uma cópia da primeira linha do GeoDataFrame original para manter os atributos
  gdf_union = gdf[0:1].copy()
  # Realiza a união de todas as geometrias no GeoDataFrame e atribui à geometria da linha copiada
  gdf_union['geometry'] = gdf.geometry.union_all()
  return gdf_union

In [30]:
def area_files_list(in_dir=str,out_dir=str,unify=True,crs=None,limiar=10):
  '''
  Processa todos os arquivos shapefile e KML/KMZ em um diretório de entrada,
  calcula suas áreas usando a função `area_calc`, e salva os resultados
  como novos shapefiles em um diretório de saída.

  Parâmetros:
  ----------
  in_dir : str
      O caminho para o diretório contendo os arquivos de entrada (shapefiles, KML/KMZ).
  out_dir : str
      O caminho para o diretório onde os arquivos de saída (shapefiles com áreas calculadas)
      serão salvos.
  unify: bool, opcional
      Se verdadeiro, as geometrias são unificadas, mantendo os atributos da primeira feição
      antes de calcular as áreas.
      Por padrão é True.
  crs : str, opcional
      O sistema de referência de coordenadas (CRS) a ser usado para o cálculo de área.
      Passado diretamente para `area_calc`. Se `None`, 'Albers IBGE SIRGAS 2000' é usado.
  limiar : int ou float, opcional
      O limiar percentual (entre 0-100) para calcular as áreas máximas e mínimas, passado para `area_calc`.
      Por padrão é 10 (10%).
  '''

  # Encontra todos os arquivos shapefile (*.shp) no diretório de entrada
  list_shapefiles = glob(os.path.join(in_dir,"*.shp"))
  print(f"{len(list_shapefiles)} shapefiles encontrados.\n")

  # Encontra todos os arquivos KML/KMZ (*.km* - .kml, .kmz) no diretório de entrada
  list_kml = glob(os.path.join(in_dir,"*.km*"))
  print(f"{len(list_kml)} kml ou kmz encontrados.\n")

  # Combina as listas de shapefiles e KML/KMZ para processamento
  list_files = list_shapefiles + list_kml

  # Garante que o diretório de saída exista; se não, ele é criado
  os.makedirs(out_dir, exist_ok=True)

  # Itera sobre cada arquivo na lista para processamento
  for file in list_files:
    # Extrai o nome do arquivo e o nome base (sem extensão)
    name = os.path.split(file)[-1]
    out_name = name.split(".")[0]
    print(f"Processando {name}")

    # Lê o arquivo geoespacial em um GeoDataFrame
    gdf = gpd.read_file(file)

    if unify:
      gdf = unify_geometries(gdf)
      print("Poligonos unificados em uma único multipolígono")
    # Chama a função area_calc para calcular as áreas e limiares
    gdf_area = area_calc(gdf=gdf,crs=crs,limiar=limiar)

    # Salva o GeoDataFrame resultante com as áreas em um novo shapefile no diretório de saída
    gdf_area.to_file(os.path.join(out_dir,f"{out_name}.shp"))
    print(f"Arquivo {out_name}.shp criado em {out_dir}.\n")

  print("Processamento concluído")

# Aplicação das funções

## copiar caminhos no prompt

In [32]:
in_dir = input("Digite o caminho para o diretório de entrada: ")
out_dir = input("Digite o caminho para o diretório de saída: ")
area_files_list(in_dir,out_dir)

Digite o caminho para o diretório de entrada: /content/iphan
Digite o caminho para o diretório de saída: /content/iphan/out
2 shapefiles encontrados.

0 kml ou kmz encontrados.

Processando aid.shp
Poligonos unificados em uma único multipolígono
Projeção utilizada: Albers IBGE SIRGAS 2000
Área calculada em hectares e limiares máximos e mínimos aplicados com 10% de variação.
Arquivo aid.shp criado em /content/iphan/out.

Processando ada.shp
Poligonos unificados em uma único multipolígono
Projeção utilizada: Albers IBGE SIRGAS 2000
Área calculada em hectares e limiares máximos e mínimos aplicados com 10% de variação.
Arquivo ada.shp criado em /content/iphan/out.

Processamento concluído


## definir caminhos fixos

In [ ]:
#Digite o caminho para o diretório de entrada e sáida
in_dir = "/content/in"
out_dir = "/content/out"
area_files_list(in_dir,out_dir)

#